In [1]:
import torch
from mmengine.config import Config

import sys
sys.path.append('/home/firdavs/surgery/surgical_fb_generation/SurgVLP')
import surgvlp

device = "cuda" if torch.cuda.is_available() else "cpu"
configs = Config.fromfile('/home/firdavs/surgery/surgical_fb_generation/SurgVLP/tests/config_peskavlp.py', lazy_import=False)['config']
model, preprocess = surgvlp.load(configs.model_config, device=device, strict_load_state_dict=False)

/home/firdavs/miniconda3/envs/firdavs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/firdavs/miniconda3/envs/firdavs/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/firdavs/miniconda3/envs/firdavs/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [2]:
model

PeskaVLP(
  (backbone_img): ImageEncoder(
    (model): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU

In [102]:
import os
import cv2
from PIL import Image

data_dir = "../../../clips_with_wiggle/fb_clips_wiggle"

for file in os.listdir(data_dir):
    i = 0
    if file.endswith(".avi"):
        i += 1
        print(f"Processing file {i}: {file}")
        file_path = os.path.join(data_dir, file)
        cap = cv2.VideoCapture(file_path)

        # Check if the video was successfully opened
        if not cap.isOpened():
            print(f"Error opening video file {file}")
            continue

        # Get total number of frames in the video
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        frames = []

        for frame_idx in range(total_frames):
            ret, frame = cap.read()
            # print(frame.shape)
            
            if not ret:
                print(f"Error reading frame {frame_idx} from {file}")
                break
            
            # Convert the frame to PIL Image object
            frame_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            frames.append(preprocess(frame_pil))

        cap.release()

        if not frames:
            print(f"No frames extracted from {file}")
            continue

        # Stack all frames into a single tensor
        frames_tensor = torch.stack(frames).to(device)  # Shape: (total_frames, C, H, W)

        with torch.no_grad():
            image_features = model(frames_tensor, None, mode='video')['img_emb']  # (total_frames, 768)
        break

Processing file 1: c9_s0_1-48-10.avi


In [112]:
preprocess(frame_pil).shape, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).transpose(2, 0, 1).shape

(torch.Size([3, 224, 224]), (3, 250, 320))

In [106]:
cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).shape

(250, 320, 3)

In [3]:
def load_clip_frames(path: str, preprocess):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print(f"Error opening video from {path}")
        return
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    for frame_idx in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            print(f"Error reading frame {frame_idx} from {path}")
            break
        frame_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        frames.append(preprocess(frame_pil))
    return frames

path = os.path.join(data_dir, "c9_s0_1-48-10.avi")
frames = load_clip_frames(path, preprocess)

In [12]:
frames_tensor = torch.stack(frames)
frames_tensor = frames_tensor.numpy().transpose(1, 0, 2, 3)
frames_tensor.shape

(3, 50, 224, 224)

In [123]:
from transformers import AutoImageProcessor, VideoMAEForPreTraining, VideoMAEModel, VideoMAEForVideoClassification
import torch

model = VideoMAEForPreTraining.from_pretrained('/home/firdavs/surgery/surgical_fb_generation/SurgicalFeedbackGeneration/checkpoints/VideoMAE-urology-pretrain-from_Arushi/').to('cuda')
preprocessor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")

In [119]:
model.videomae

VideoMAEModel(
  (embeddings): VideoMAEEmbeddings(
    (patch_embeddings): VideoMAEPatchEmbeddings(
      (projection): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
  )
  (encoder): VideoMAEEncoder(
    (layer): ModuleList(
      (0-11): 12 x VideoMAELayer(
        (attention): VideoMAESdpaAttention(
          (attention): VideoMAESdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=False)
            (key): Linear(in_features=768, out_features=768, bias=False)
            (value): Linear(in_features=768, out_features=768, bias=False)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): VideoMAESelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): VideoMAEIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn)

In [120]:
with open('/home/firdavs/surgery/surgical_fb_generation/SurgicalFeedbackGeneration/checkpoints/VideoMAE-CholecT45-seed=42-dt=2025_03_08.20_30_38/pretrain/epoch_50.pth', 'rb') as f:
    epoch = torch.load(f, weights_only=False)
    state_dict = epoch['model_state_dict']

In [125]:
model.load_state_dict(state_dict, strict=False)

_IncompatibleKeys(missing_keys=['videomae.layernorm.weight', 'videomae.layernorm.bias'], unexpected_keys=[])

In [127]:
model = model.videomae
model

VideoMAEModel(
  (embeddings): VideoMAEEmbeddings(
    (patch_embeddings): VideoMAEPatchEmbeddings(
      (projection): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
  )
  (encoder): VideoMAEEncoder(
    (layer): ModuleList(
      (0-11): 12 x VideoMAELayer(
        (attention): VideoMAESdpaAttention(
          (attention): VideoMAESdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=False)
            (key): Linear(in_features=768, out_features=768, bias=False)
            (value): Linear(in_features=768, out_features=768, bias=False)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): VideoMAESelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): VideoMAEIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn)

In [129]:
from pytorchvideo.data.encoded_video import EncodedVideo
from pytorchvideo.transforms import UniformTemporalSubsample

def extract_features(video_path, preprocessor):
    # Load video
    video = EncodedVideo.from_path(video_path)

    # Get clip from the video
    video_data = video.get_clip(start_sec=0, end_sec=10.0)["video"]
    print(video_data.shape)
    # Subsample frames
    num_frames = 16
    subsampler = UniformTemporalSubsample(num_frames)
    subsampled_frames = subsampler(video_data)
    video_data_np = subsampled_frames.numpy().transpose(1, 0, 2, 3)
    print(video_data_np.shape)
    # Preprocess video data
    frame_list = list(video_data_np)
    video_batch = [frame_list]
    inputs = preprocessor.preprocess(video_batch, return_tensors="pt").to(device)
    
    # Obtain embedding
    with torch.no_grad():
        outputs = model(inputs.pixel_values)
        last_embed = outputs.last_hidden_state  # Shape: (batch_size, num_frames, hidden_dim)
        print(last_embed.shape)
    
    return last_embed

emb = extract_features(os.path.join(data_dir, file), preprocessor)

torch.Size([3, 49, 250, 320])
(16, 3, 250, 320)
torch.Size([1, 1568, 768])


In [130]:
emb.squeeze().cpu().numpy().shape

(1568, 768)

In [89]:
1568 / 16

98.0